# 00. JupyterLab 환경과 재현 가능한 실행 습관

이 노트북은 앞으로 사용할 **프로젝트 경로, Python 커널, 패키지 버전, 난수 시드**를 점검합니다.
AI 엔지니어의 실험은 “한 번 성공한 코드”보다 **다시 실행해도 같은 근거를 남기는 코드**가 중요합니다.

학습 목표:

- Jupyter의 셀과 커널 상태를 구분한다.
- 현재 노트북이 올바른 `.venv` 커널에서 실행되는지 확인한다.
- 프로젝트의 `src/`, `data/`, `artifacts/` 경로를 안전하게 찾는다.
- Python/NumPy/PyTorch 난수를 고정하고 재현성을 확인한다.

## 권장 실행 순서

1. JupyterLab에서 이 파일을 연다.
2. 우측 상단 커널이 `Python (LLM Engineering Lab)`인지 확인한다.
3. `Kernel → Restart Kernel and Run All Cells`로 전체 실행한다.
4. 오류가 없으면 `01`부터 번호 순서대로 진행한다.

노트북은 설명과 관찰에, `src/`는 재사용 가능한 구현에, `tests/`는 자동 검증에 사용합니다.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """현재 폴더부터 위로 올라가며 프로젝트 루트를 찾습니다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "pyproject.toml과 src/가 있는 프로젝트 폴더를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("project:", PROJECT_ROOT)
print("python :", sys.executable)

## 커널과 패키지 상태 확인

JupyterLab 서버를 실행한 Python과 **현재 노트북 커널의 Python은 다를 수 있습니다.**
아래의 `sys.executable`이 프로젝트의 `.venv\Scripts\python.exe`를 가리키는지 확인하세요.

In [ ]:
import platform
import importlib.metadata as metadata

packages = ["numpy", "pandas", "scikit-learn", "torch"]
print("Python   :", platform.python_version())
print("OS       :", platform.platform())
for package in packages:
    try:
        print(f"{package:12}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:12}: 설치되지 않음")

## 셀 상태(state)를 직접 관찰하기

커널은 셀 실행이 끝나도 메모리의 변수를 유지합니다. 아래 두 셀을 순서대로 실행한 뒤,
두 번째 셀만 여러 번 실행해 보세요. 실행 순서에 따라 값이 달라지는 것이 노트북의 대표적인 함정입니다.

In [ ]:
experiment_state = {"run_count": 0, "losses": []}
experiment_state

In [ ]:
experiment_state["run_count"] += 1
experiment_state["losses"].append(round(1 / experiment_state["run_count"], 3))
experiment_state

셀 번호가 `[1]`, `[2]`, `[7]`처럼 뒤섞였거나 변수가 어디서 만들어졌는지 불분명하면 커널을 재시작하세요.
제출하거나 공유하기 전에는 반드시 **Restart Kernel and Run All Cells**로 검증합니다.

## 난수와 재현성

모델 초기값, 데이터 셔플, 샘플링에는 난수가 사용됩니다. 같은 시드를 사용하면 디버깅과 실험 비교가 쉬워집니다.
완전 결정론적 GPU 연산은 속도를 낮출 수 있으므로 목적에 맞게 선택해야 합니다.

In [ ]:
import random
import numpy as np

from llm_engineering_lab.config import seed_everything

seed_everything(42)
python_sample = [random.random() for _ in range(3)]
numpy_sample = np.random.random(3)

print("Python:", python_sample)
print("NumPy :", numpy_sample)

In [ ]:
# 같은 시드로 되돌리면 같은 수열이 다시 나와야 합니다.
seed_everything(42)
assert python_sample == [random.random() for _ in range(3)]
assert np.allclose(numpy_sample, np.random.random(3))
print("재현성 검사 통과")

## 프로젝트 경로와 패키지 import

현재 작업 폴더(`cwd`)를 문자열로 하드코딩하지 않고 프로젝트 루트를 기준으로 경로를 만듭니다.
그러면 JupyterLab을 프로젝트 루트 또는 `notebooks/`에서 열어도 같은 코드가 동작합니다.

In [ ]:
from llm_engineering_lab.config import ProjectPaths

paths = ProjectPaths.discover()
assert paths.root == PROJECT_ROOT
assert paths.data.is_dir()

print("data     :", paths.data)
print("artifacts:", paths.artifacts)

In [ ]:
from llm_engineering_lab.data import load_ticket_data

tickets = load_ticket_data(paths.data / "customer_support_tickets.csv")
print("shape:", tickets.shape)
tickets.head(3)

## TODO: 커널 건강 상태 함수 따라 치기

아래 함수는 실험 시작 전에 자주 확인할 정보를 한 딕셔너리로 모읍니다.
구현을 지운 뒤 주석만 보고 다시 작성해 보세요.

In [ ]:
def kernel_health_report(project_root: Path) -> dict[str, object]:
    # TODO 1: Python 실행 파일, 작업 폴더, 프로젝트 존재 여부를 반환하세요.
    return {
        "python": sys.executable,
        "cwd": str(Path.cwd()),
        "project_exists": project_root.is_dir(),
        "data_exists": (project_root / "data").is_dir(),
    }


health = kernel_health_report(PROJECT_ROOT)
assert health["project_exists"] is True
assert health["data_exists"] is True
assert str(PROJECT_ROOT) in str(paths.root)
health

## 실험 노트북 운영 규칙

- 원본 데이터는 덮어쓰지 않고 가공 결과를 별도 변수/파일에 저장합니다.
- 긴 학습은 함수나 `scripts/`로 옮기고 노트북에서는 작은 설정으로 검증합니다.
- 지표와 그림만 보지 말고 시드, 데이터 분할, 하이퍼파라미터도 기록합니다.
- 비밀키를 셀에 직접 쓰지 않습니다. 환경 변수나 비밀 관리 도구를 사용합니다.
- 셀을 반복 실행해도 파일이나 상태가 망가지지 않는 **멱등성**을 지향합니다.

In [ ]:
# 마지막 셀: 최소 환경 검증
required_paths = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "tests",
]
assert all(path.exists() for path in required_paths)
assert len(tickets) > 0
print("✅ 00 노트북 준비 완료 — 다음은 01 중급 Python 패턴입니다.")